# Session 3: Advanced: scaling out and running in production

*LLM Inference Engineering*, session 3 of 3. Parallelism built by hand on the CPU so you can check the arithmetic, then the commands that run real models across GPUs and nodes, and the operational side of keeping them up.

## How the series is divided

| Session | The question it answers | Topics |
|---|---|---|
| **1. Basics** | What happens between a prompt and its answer, and why is it slow? | Tokens, prefill and decode, and the KV cache, built from scratch; Memory math: weights, KV cache, what fits on a GPU; The GPU underneath: memory hierarchy, roofline, the decode speed limit; The metrics: TTFT, TPOT, throughput, goodput, cost per token; First steps with vLLM: offline engine, sampling, the OpenAI server, a load test |
| **2. Intermediate** | Which tricks make serving fast, and what does each one actually do? | Continuous batching and PagedAttention: a scheduler and a block allocator in code; Prefix caching and chunked prefill; Kernels: FlashAttention, CUDA graphs, the roofline measured; Quantization: INT8, FP8, INT4, NF4, AWQ, GPTQ, MXFP4, built and compared; Speculative decoding: draft-and-verify from scratch, n-gram, EAGLE-3, MTP; Structured outputs, scoring, embeddings, LoRA; vLLM vs SGLang |
| **3. Advanced** *(this one)* | How do you serve a model bigger than one GPU, reliably, for many users? | How GPUs connect: NVLink, PCIe, InfiniBand, NCCL, measured; Tensor, pipeline, data and expert parallelism in code, and when to use which; Big models: 70B to DeepSeek and Kimi on 8 GPUs, multi-node; Disaggregated prefill and decode, KV offload, TensorRT-LLM and Dynamo; Hosting open-weight models: recipes, tool calling, reasoning, vision, local machines; Production: Docker and Kubernetes, metrics, priorities, the tuning playbook |

| | |
|---|---|
| **Needs** | A laptop for the from-scratch parallelism section; 2+ NVIDIA GPUs for the multi-GPU measurements; one GPU for the hosting sections; 8-GPU nodes for the commands marked *(reading)*. |
| **Labels** | *(no GPU needed)* runs on a laptop; *(reading)* has nothing to execute; everything else needs an NVIDIA GPU |
| **Run order** | Top to bottom. Engines are started and stopped per section, so you can also jump to one after running the setup cell. |

In [ ]:
%pip install -q vllm openai matplotlib pandas transformers huggingface_hub pyarrow

In [ ]:
# Shared helpers live in infer_utils.py next to this notebook:
# MODEL, MAX_LEN, PORT, BASE, launch, vllm_serve, stop, shutdown, measure_stream, bench_openai, gpu_mem
from infer_utils import *
try:
    from vllm import LLM, SamplingParams
except ImportError:
    print("vLLM is not installed here; the from-scratch and reading sections still run.")
import os, gc, time, json, subprocess
import numpy as np, pandas as pd, requests
from openai import OpenAI, AsyncOpenAI
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)          # the default model's tokenizer, used for token counts
client = OpenAI(base_url=f"{BASE}/v1", api_key="not-needed")
results = {}                                         # benchmark tables, keyed by engine
print(env_report())

## Contents

1. Parallelism from scratch: split a layer across "GPUs" and check the answer *(no GPU needed)*
2. Connecting GPUs: check the topology, measure the links *(needs 2+ GPUs)*
3. Scaling out: beyond one GPU *(reading)*
4. Sizing: which GPUs, which parallelism *(no GPU needed)*
5. The giant MoEs on one 8-GPU node *(reading: needs 8x H200 / B200)*
6. Multi-node *(reading)*
7. KV cache beyond one GPU: offload, LMCache and disaggregated prefill
8. TensorRT-LLM offline `LLM` API
9. `trtllm-serve` + the same benchmark harness
10. Head-to-head on your GPU
11. How to read a vLLM recipe *(reading)*
12. Tool calling end to end *(runnable)*
13. Reasoning models: separate the thinking from the answer *(runnable)*
14. Vision-language models *(runnable)*
15. gpt-oss: the recipe in practice *(runnable on 24 GB+ for 20b, 80 GB for 120b)*
16. Packaging it: Docker, config files, keys and probes *(reading)*
17. Operating it: priorities, metrics and a launch checklist
18. Sleep mode: give the GPU back without killing the engine
19. `AsyncLLM`: the engine inside the server, in your own process
20. Reaching into the workers: `apply_model` and `collective_rpc`
21. Batch jobs: `vllm run-batch` and data parallelism
22. Profiling: where does a step actually go? *(reading)*
23. A planner that reads any Hugging Face config *(no GPU needed)*
24. The local toolbox *(reading)*
25. vLLM when the model *almost* fits
26. Edge-case checklist: errors you will hit *(reading)*
27. What practitioners report (treat as anecdotes, not benchmarks) *(reading)*

## 1. Parallelism from scratch: split a layer across "GPUs" and check the answer *(no GPU needed)*

We simulate N GPUs as N shards on the CPU. Everything here is real arithmetic, so you can verify that each scheme computes the same result as one big GPU, and count what has to cross the wire.

**Tensor parallel (Megatron-style) for an MLP block** `Y = act(X @ W1) @ W2`:
- Split `W1` by **columns**: each GPU computes its own slice of the hidden activations, no communication needed.
- Split `W2` by **rows**: each GPU produces a partial `Y`; one **all-reduce** (sum) combines them.

One all-reduce per MLP and one per attention block, every layer, every token. That's why TP needs NVLink.

In [ ]:
torch.manual_seed(0)
B, D, Hdim, N = 8, 1024, 4096, 4                    # batch, model dim, MLP hidden dim, number of GPUs
X = torch.randn(B, D); W1 = torch.randn(D, Hdim) / D**.5; W2 = torch.randn(Hdim, D) / Hdim**.5
act = torch.nn.functional.silu
Y_full = act(X @ W1) @ W2                           # one big GPU

W1_shards = W1.chunk(N, dim=1)                      # column-parallel
W2_shards = W2.chunk(N, dim=0)                      # row-parallel
partials = [act(X @ w1) @ w2 for w1, w2 in zip(W1_shards, W2_shards)]   # each GPU, independently
Y_tp = sum(partials)                                 # the all-reduce

print("TP result matches:", torch.allclose(Y_full, Y_tp, atol=1e-4))
print(f"per-GPU weight memory: {(W1.numel() + W2.numel()) * 2 / N / 2**20:.1f} MiB of {(W1.numel() + W2.numel()) * 2 / 2**20:.1f} MiB")
print(f"communication per MLP: all-reduce of {B * D * 2 / 1024:.0f} KiB (batch x model dim, BF16)")

**Pipeline parallel**: each GPU holds a contiguous range of layers and passes activations to the next. Communication is small (one hand-off per stage boundary), so it works across nodes. The cost is the **bubble**: while the first micro-batch works through the pipeline, later stages wait.

**Expert parallel** (MoE): each GPU holds some of the experts. A router sends each token to its top-k experts, wherever they live (an **all-to-all**), and the results come back. Memory scales with total experts, compute with active ones.

In [ ]:
# Pipeline: verify the layered result and compute the bubble for a few schedules
layers = [torch.nn.Linear(D, D) for _ in range(8)]
x = torch.randn(B, D)
full = x
for l in layers: full = torch.relu(l(full))
stages = [layers[0:2], layers[2:4], layers[4:6], layers[6:8]]          # 4 GPUs, 2 layers each
h = x
for st in stages:                                                       # GPU i -> GPU i+1 hand-off
    for l in st: h = torch.relu(l(h))
print("PP result matches:", torch.allclose(full, h, atol=1e-5))
for m in (1, 4, 16):
    p = len(stages)
    print(f"  {m:2} micro-batches on {p} stages: bubble = (p-1)/(m+p-1) = {(p-1)/(m+p-1):.0%} idle")

# Expert parallel: top-2 routing of 64 tokens over 8 experts on 4 GPUs (2 experts each)
E, GPUS, TOPK = 8, 4, 2
router = torch.randn(D, E)
tokens = torch.randn(64, D)
top = (tokens @ router).topk(TOPK, dim=-1).indices                     # each token picks 2 experts
owner = top // (E // GPUS)                                               # which GPU owns each chosen expert
per_gpu = torch.bincount(owner.flatten(), minlength=GPUS)
print(f"EP: token-expert assignments per GPU {per_gpu.tolist()} (ideal {64*TOPK//GPUS} each). "
      "Imbalance is why vLLM has --enable-eplb.")

In [ ]:
# What the wire costs: ring all-reduce time = 2 (N-1)/N x bytes / link bandwidth
def allreduce_us(bytes_, n, gbps):
    return 2 * (n - 1) / n * bytes_ / (gbps * 1e9) * 1e6

hidden, layers_70b = 8192, 80
msg = 1 * hidden * 2                                                    # one token's activations, BF16
links = {"NVLink 4 (H100, 900 GB/s)": 450, "PCIe Gen5 x16 (64 GB/s)": 32, "InfiniBand NDR 400 Gb/s (50 GB/s)": 50}
for name, gbps in links.items():                                        # ~half the bidirectional figure per direction
    per_token_ms = 2 * layers_70b * allreduce_us(msg * 64, 8, gbps) / 1e3   # batch 64, 2 all-reduces per layer
    print(f"{name:36} TP=8 all-reduce time per decode step at batch 64: {per_token_ms:6.2f} ms")
print("A 70B decode step on 8 H100s takes ~10-15 ms; communication must stay a small fraction of that.")

(These are bandwidth-only estimates; real all-reduces for small messages are dominated by latency, which makes slow links look even worse.)

**When to use which**

| Your situation | Use | vLLM flags |
|---|---|---|
| Model fits on one GPU, need more throughput | **Data parallel**: independent replicas | `--data-parallel-size N` (or N separate servers behind a load balancer) |
| Doesn't fit on one GPU, fits in one NVLink node | **Tensor parallel** inside the node | `--tensor-parallel-size 2/4/8` |
| Doesn't fit in one node | **TP inside, PP across** nodes | `-tp 8 -pp 2` |
| Big MoE, many concurrent users | **DP attention + expert parallel** | `-dp 8 --enable-expert-parallel` |
| Consumer GPUs over PCIe, no NVLink | **Pipeline** before tensor parallel | `--pipeline-parallel-size 2` |
| Very long contexts | Context parallel for decode | `--decode-context-parallel-size N` |

## 2. Connecting GPUs: check the topology, measure the links *(needs 2+ GPUs)*

Before choosing a parallelism, find out how your GPUs are wired. `nvidia-smi topo -m` prints a matrix: `NV#` means NVLink (good for TP), `PIX`/`PXB` a PCIe switch, `SYS` a path through the CPU (slowest).

In [ ]:
import shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found: no NVIDIA driver on this machine")
NGPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("GPUs visible:", NGPU)

Every multi-GPU engine talks through **NCCL**. This script measures all-reduce bus bandwidth across message sizes, the same number `nccl-tests` reports. Run it with `torchrun`, one process per GPU.

In [ ]:
SCRIPT = r'''
import os, time, torch, torch.distributed as dist
dist.init_process_group("nccl")
rank, world = dist.get_rank(), dist.get_world_size()
torch.cuda.set_device(rank % torch.cuda.device_count())
for mb in (0.0625, 1, 16, 256):
    x = torch.ones(int(mb * 2**20 // 2), dtype=torch.bfloat16, device="cuda")
    for _ in range(5): dist.all_reduce(x)
    torch.cuda.synchronize(); t0 = time.perf_counter(); iters = 20
    for _ in range(iters): dist.all_reduce(x)
    torch.cuda.synchronize(); dt = (time.perf_counter() - t0) / iters
    busbw = 2 * (world - 1) / world * x.numel() * 2 / dt / 1e9
    if rank == 0: print(f"{mb:8.4g} MiB  {dt*1e6:9.1f} us  bus bandwidth {busbw:7.1f} GB/s")
dist.destroy_process_group()
'''
open("allreduce_bench.py", "w").write(SCRIPT)
if NGPU >= 2:
    r = subprocess.run(["torchrun", f"--nproc_per_node={NGPU}", "allreduce_bench.py"], capture_output=True, text=True,
                       env={**os.environ, "NCCL_DEBUG": "WARN"})
    print(r.stdout or r.stderr[-2000:])
else:
    print("Needs 2+ GPUs. On an 8x H100 node expect ~350-450 GB/s bus bandwidth at 256 MiB over NVLink; "
          "PCIe-only consumer cards land around 10-25 GB/s.")

NCCL settings you will actually touch:

| Variable | When |
|---|---|
| `NCCL_DEBUG=INFO` | First thing when multi-GPU hangs or is slow: it prints which transport each link uses (`NVLS`, `P2P`, `NET/IB`, `NET/Socket`) |
| `NCCL_P2P_DISABLE=1` | Consumer GPUs that hang on peer-to-peer over PCIe |
| `NCCL_IB_HCA=mlx5` / `NCCL_SOCKET_IFNAME=eth0` | Multi-node: pin the right InfiniBand card or network interface |
| `--ipc=host` / `--shm-size` (Docker) | Required for multi-process engines inside containers |

Now the same comparison an engineer makes before deploying: one model, two GPUs, **TP=2** (one server, split model) versus **DP=2** (two replicas). TP wins on per-user latency, DP on total throughput when the model fits on one GPU.

In [ ]:
if NGPU >= 2:
    res = {}
    for name, flags in [("TP=2", ["--tensor-parallel-size", "2"]), ("DP=2", ["--data-parallel-size", "2"])]:
        srv = vllm_serve(*flags, logfile=f"vllm_{name}.log")
        res[name] = await bench_openai(BASE, MODEL, levels=(1, 16, 64))
        stop(srv)
    display(pd.concat(res, names=["layout"]).reset_index(level=0))
else:
    print("Needs 2+ GPUs.")

## 3. Scaling out: beyond one GPU *(reading)*

These need multiple GPUs, so they're shown as commands, not run.

### Tensor / pipeline / data / expert parallelism
```bash
# 70B model on 4 GPUs in one node (NVLink): split every layer → lower latency
vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 4

# 2 nodes × 8 GPUs: TP inside the node, PP across nodes
vllm serve <model> --tensor-parallel-size 8 --pipeline-parallel-size 2

# Throughput: independent replicas (or put N servers behind a load balancer)
vllm serve <model> --data-parallel-size 4

# Large MoE (DeepSeek / Qwen-MoE): spread experts across GPUs
vllm serve <moe-model> --tensor-parallel-size 8 --enable-expert-parallel
```

| Strategy | Splits | Good for | Watch out |
|---|---|---|---|
| TP | each layer's matrices | latency, models that don't fit | all-reduce every layer → needs NVLink |
| PP | layers into stages | crossing nodes | doesn't cut single-request latency |
| DP | whole replicas | throughput | memory per replica |
| EP | MoE experts | huge MoE models | all-to-all traffic, expert load balance |

### Disaggregated prefill / decode
Prefill (compute-bound) and decode (bandwidth-bound) run on **separate GPU pools**; the KV cache is shipped between them over NVLink/RDMA (e.g. NIXL). vLLM supports this via KV-transfer connectors, and orchestration layers such as **NVIDIA Dynamo** and **llm-d** manage it on Kubernetes, adding **KV-cache-aware routing** (send each request to the replica that already holds its prefix).

```
          ┌──────────── prefill pool (TTFT) ────────────┐        ┌────── decode pool (TPOT) ──────┐
request ─▶│ GPU  GPU  GPU  GPU   → builds KV cache        │──KV──▶ │ GPU  GPU  GPU  GPU → tokens    │─▶ stream
          └──────────────────────────────────────────────┘ NIXL   └────────────────────────────────┘
```
> Disaggregation pays off at scale with long prompts and strict SLOs on both TTFT and TPOT. For small deployments, **chunked prefill** (on by default) gets you most of the benefit for free.

## 4. Sizing: which GPUs, which parallelism *(no GPU needed)*

Before any launch command, do the arithmetic from §23. The decision tree the recipes follow:

1. **Weights + KV for your context fit on one GPU?** Single GPU. Add GPUs as **data-parallel replicas** (`--data-parallel-size N`) for throughput.
2. **Fit on one node (8 GPUs)?** Tensor parallel inside the node (`-tp 8`). For MoE, consider **DP attention + expert parallel** (`-dp 8 --enable-expert-parallel`): attention is replicated per GPU and experts are spread, which avoids TP's all-reduce on the attention path and suits high-concurrency serving.
3. **Bigger than a node?** TP inside the node, pipeline parallel across nodes (`-tp 8 -pp 2`), or wide EP across nodes for the largest MoEs.
4. **Long context dominates memory?** FP8 KV cache (`--kv-cache-dtype fp8`), then context parallelism for decode (`--decode-context-parallel-size`), then KV offload (§7).

In [ ]:
# Weight memory at the precision each recipe recommends, against common nodes
models = {"gpt-oss-120b (MXFP4)": (117, 0.53), "Qwen3-235B-A22B (FP8)": (235, 1.0), "Llama-3.3-70B (FP8)": (70, 1.0),
          "DeepSeek-V3.2 (FP8)": (671, 1.0), "Kimi K2 Thinking (INT4)": (1000, 0.5)}
nodes = {"1x H100 80GB": 80, "8x H100 80GB": 640, "8x H200 141GB": 1128, "8x B200 192GB": 1536}
rows = []
for m, (params_b, bytes_per) in models.items():
    w = params_b * bytes_per
    rows.append({"model": m, "weights_GB": round(w), **{n: ("fits" if w * 1.25 < cap else "no") for n, cap in nodes.items()}})
display(pd.DataFrame(rows))
print("'fits' leaves 20% of memory for KV cache and activations. Check the recipe's vram_minimum_gb before buying.")

## 5. The giant MoEs on one 8-GPU node *(reading: needs 8x H200 / B200)*

These are the commands from the recipes, with what each flag is for.

```bash
# DeepSeek-V3.2: TP 8, sparse-attention tokenizer mode, tool + reasoning parsers
vllm serve deepseek-ai/DeepSeek-V3.2 \
  --tensor-parallel-size 8 --trust-remote-code \
  --tokenizer-mode deepseek_v32 --tool-call-parser deepseek_v32 --enable-auto-tool-choice \
  --reasoning-parser deepseek_v3

# Same model, high-concurrency layout: attention data-parallel, experts spread across GPUs
vllm serve deepseek-ai/DeepSeek-V3.2 -dp 8 --enable-expert-parallel

# Add DeepSeek's own multi-token-prediction heads as the speculator
  --speculative-config '{"method":"mtp","num_speculative_tokens":3}'

# Kimi K2 Thinking (~1T total): TP 8; DCP 8 shards the KV cache for long contexts
vllm serve moonshotai/Kimi-K2-Thinking \
  --tensor-parallel-size 8 --decode-context-parallel-size 8 \
  --enable-auto-tool-choice --tool-call-parser kimi_k2 --reasoning-parser kimi_k2 \
  --trust-remote-code

# Qwen3-235B-A22B in FP8 on 4-8 GPUs
vllm serve Qwen/Qwen3-235B-A22B-FP8 --tensor-parallel-size 4 \
  --max-model-len 32768 --max-num-batched-tokens 32768 --gpu-memory-utilization 0.8
```

Flags that matter at this scale:

| Flag | Why |
|---|---|
| `--enable-expert-parallel` | Spread experts across GPUs instead of slicing every expert (TP). Less communication per token for MoE. |
| `--enable-eplb` | Expert-parallel load balancer: replicates hot experts so one GPU isn't the bottleneck |
| `--all2all-backend` | The kernel library for EP's token shuffle (DeepEP and others); matters most across nodes |
| `--async-scheduling` | Overlaps CPU scheduling with GPU execution; the gpt-oss recipe enables it for throughput |
| `--load-format runai_streamer` | Streams safetensors from disk or S3 in parallel; cuts cold start for 600 GB checkpoints |
| `--max-num-batched-tokens` | The chunked-prefill budget (Session 2 §6). Larger favours throughput, smaller protects TPOT. |

## 6. Multi-node *(reading)*

The docs' rule: **tensor parallel = GPUs per node, pipeline parallel = number of nodes.** Two runtimes:

```bash
# multiprocessing, no Ray: run on every node, node-rank 0 is the head
vllm serve /models/big --tensor-parallel-size 8 --pipeline-parallel-size 2 \
  --nnodes 2 --node-rank 0 --master-addr $HEAD_IP
vllm serve /models/big --tensor-parallel-size 8 --pipeline-parallel-size 2 \
  --nnodes 2 --node-rank 1 --master-addr $HEAD_IP --headless

# Ray: start the cluster with examples/ray_serving/run_cluster.sh on each node, then from the head
vllm serve /models/big --tensor-parallel-size 8 --pipeline-parallel-size 2 --distributed-executor-backend ray
```

Things that bite: every node needs the **same** image and model path; InfiniBand or RoCE must actually be used by NCCL (check with `NCCL_DEBUG=INFO`, look for `NET/IB`); and TP across nodes over Ethernet is slow enough that PP across nodes almost always wins.

## 7. KV cache beyond one GPU: offload, LMCache and disaggregated prefill

Three steps up in complexity, all through vLLM's KV connector interface.

**1. Native CPU offload** (one flag). Evicted prefix blocks go to CPU RAM instead of being dropped, so a returning conversation reloads its KV over PCIe instead of re-running prefill. Worth it for chat and agents with long, repeated contexts.

In [ ]:
server = vllm_serve("--kv-offloading-size", "16", "--kv-offloading-backend", "native")   # 16 GiB of CPU RAM for KV
long_ctx = "Background: " + "The cluster runs H100s with NVLink. " * 600
q = lambda i: [{"role": "system", "content": long_ctx}, {"role": "user", "content": f"Question {i}: summarise in 5 words."}]
cold = measure_stream(q(0), max_tokens=8)["ttft_ms"]
warm = measure_stream(q(1), max_tokens=8)["ttft_ms"]
print(f"cold TTFT {cold:.0f} ms, warm TTFT {warm:.0f} ms (prefix from GPU cache)")
requests.post(f"{BASE}/reset_prefix_cache")          # needs VLLM_SERVER_DEV_MODE=1; ignore a 404 otherwise
print(f"after GPU cache reset: {measure_stream(q(2), max_tokens=8)['ttft_ms']:.0f} ms (served from CPU offload if the reset was allowed)")
stop(server)

**2. LMCache** (`--kv-offloading-backend lmcache`, or the `LMCacheConnectorV1` connector): adds disk and remote tiers and shares KV across vLLM instances, so a prefix computed on replica A is a cache hit on replica B.

```python
from vllm.config import KVTransferConfig
llm = LLM(model=MODEL, kv_transfer_config=KVTransferConfig(kv_connector="LMCacheConnectorV1", kv_role="kv_both"))
```

**3. Disaggregated prefill and decode with NIXL** (two GPUs or two nodes). Prefill runs on one instance, its KV cache moves over NVLink or RDMA to a decode instance, and a proxy routes each request through both. This is the set-up from the vLLM docs:

```bash
# prefill instance, GPU 0
CUDA_VISIBLE_DEVICES=0 VLLM_NIXL_SIDE_CHANNEL_PORT=5600 vllm serve Qwen/Qwen3-0.6B --port 8100 \
  --kv-transfer-config '{"kv_connector":"NixlConnector","kv_role":"kv_producer"}'
# decode instance, GPU 1
CUDA_VISIBLE_DEVICES=1 VLLM_NIXL_SIDE_CHANNEL_PORT=5601 vllm serve Qwen/Qwen3-0.6B --port 8200 \
  --kv-transfer-config '{"kv_connector":"NixlConnector","kv_role":"kv_consumer"}'
# proxy (from the vLLM repo)
python tests/v1/kv_connector/nixl_integration/toy_proxy_server.py --port 8192 \
  --prefiller-hosts localhost --prefiller-ports 8100 --decoder-hosts localhost --decoder-ports 8200
```

Measure before adopting it: run Session 1 §15's concurrency sweep against port 8192 and against a single instance with the same two GPUs as `--data-parallel-size 2`. Disaggregation wins when long prompts are hurting TPOT for everyone else; for short prompts, DP plus chunked prefill is usually as good and much simpler.

| Piece | What it is | When you'd touch it |
|---|---|---|
| **TensorRT-LLM** | NVIDIA's engine. Since 1.0 the default backend is **PyTorch** (no separate engine-build step); the older TensorRT backend compiles a model-specific engine. Ships `LLM` API, `trtllm-serve`, `trtllm-bench` | Squeezing the last 10–30% out of NVIDIA GPUs, FP8/NVFP4 on Hopper/Blackwell |
| **Dynamo** | Datacenter-scale orchestration *on top of* vLLM, SGLang or TRT-LLM: PD disaggregation, KV-aware routing, NIXL KV transfer, KV offload | Multi-node serving, disaggregation |
| **NIM** | Pre-packaged containers (engine + tuned config per GPU) behind an OpenAI API | You want a supported, pre-tuned endpoint and have an NVIDIA AI Enterprise licence |
| **Model Optimizer** (`modelopt`) | Quantization toolkit (FP8, NVFP4, INT4 AWQ) producing checkpoints all three engines can load | Making your own FP8/FP4 checkpoint |

> **Watch out.** **Install edge case.** TRT-LLM is Linux + NVIDIA only and is pinned to specific CUDA/PyTorch versions. The path of least pain is the NGC container:
> ```bash
> docker run --rm -it --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -p 8001:8001 \
>   -v ~/.cache/huggingface:/root/.cache/huggingface nvcr.io/nvidia/tensorrt-llm/release:<tag>
> ```
> `--ipc=host` (or a large `--shm-size`) matters for every engine in Docker: multi-GPU runs crash with cryptic NCCL/shared-memory errors without it.

## 8. TensorRT-LLM offline `LLM` API

Looks almost identical to vLLM's. Run this from inside the TRT-LLM container/env (it imports `tensorrt_llm`).

In [ ]:
TRT_SNIPPET = r'''
from tensorrt_llm import LLM, SamplingParams
from tensorrt_llm.llmapi import KvCacheConfig
import time

llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct",
          kv_cache_config=KvCacheConfig(free_gpu_memory_fraction=0.8, enable_block_reuse=True))  # block reuse = prefix caching
prompts = [f"Write a haiku about GPU number {i}." for i in range(256)]
t0 = time.perf_counter()
outs = llm.generate(prompts, SamplingParams(max_tokens=64, temperature=0.8))
dt = time.perf_counter() - t0
n = sum(len(o.outputs[0].token_ids) for o in outs)
print(outs[0].outputs[0].text.strip())
print(f"{len(prompts)} prompts, {n} tokens in {dt:.1f}s -> {n/dt:.0f} tok/s")
'''
open("trtllm_offline.py", "w").write(TRT_SNIPPET)
TRTLLM_PY = os.environ.get("TRTLLM_PY")       # python inside the TRT-LLM env, if you have one
if TRTLLM_PY:
    print(subprocess.run([TRTLLM_PY, "trtllm_offline.py"], capture_output=True, text=True).stdout[-1500:])
else:
    print("Wrote trtllm_offline.py. Run it inside the TRT-LLM container:  python trtllm_offline.py")

## 9. `trtllm-serve` + the same benchmark harness

`trtllm-serve` exposes the same OpenAI API, so Session 2 §27's `bench_openai` works unchanged. Start it inside the container (or set `TRTLLM_SERVE` to the binary path):

```bash
trtllm-serve Qwen/Qwen2.5-1.5B-Instruct --host 0.0.0.0 --port 8001 \
    --max_batch_size 128 --max_num_tokens 8192 --max_seq_len 4096 \
    --kv_cache_free_gpu_memory_fraction 0.8
# extra knobs (spec decode, CUDA graphs, attention backend…) go in a YAML file:
#   --config extra.yaml        (older releases: --extra_llm_api_options extra.yaml)
```

In [ ]:
TRT_BASE = "http://localhost:8001"
trt = None
if os.environ.get("TRTLLM_SERVE"):
    trt = launch([os.environ["TRTLLM_SERVE"], MODEL, "--port", "8001", "--max_seq_len", str(MAX_LEN),
                  "--kv_cache_free_gpu_memory_fraction", "0.8"], "trtllm_server.log", f"{TRT_BASE}/health")
try:
    urllib.request.urlopen(f"{TRT_BASE}/health", timeout=2)
    results["trtllm"] = await bench_openai(TRT_BASE, MODEL); display(results["trtllm"])
except Exception as e:
    print("No TRT-LLM server on :8001, skipping.", type(e).__name__)

## 10. Head-to-head on your GPU

Re-launch vLLM with matching memory settings and benchmark it with the **same** harness, then plot all engines on the latency–throughput plane.

In [ ]:
server = launch(["vllm", "serve", MODEL, "--port", str(PORT), "--max-model-len", str(MAX_LEN),
                 "--gpu-memory-utilization", "0.80"], "vllm_server.log", f"{BASE}/health")
results["vllm"] = await bench_openai(BASE, MODEL)
server.terminate(); server.wait(60)

fig, ax = plt.subplots()
for name, df_ in results.items():
    ax.plot(df_.system_tok_s, df_.per_user_tok_s, "o-", label=name)
    for _, r in df_.iterrows():
        ax.annotate(int(r.concurrency), (r.system_tok_s, r.per_user_tok_s), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set(xlabel="system throughput (tok/s)", ylabel="per-user speed (tok/s)", title="Same model, same GPU, same harness")
ax.legend(); plt.show()
display(pd.concat(results, names=["engine"]).reset_index(level=0))

> **Watch out.** **How to read an engine comparison honestly**
> - Same model weights, same dtype, same max context, same memory fraction. Otherwise you're comparing configs, not engines.
> - Warm up first (CUDA graphs and compile happen on early requests).
> - Fixed output length (`ignore_eos`) or you're benchmarking how chatty the sampler was.
> - A 1.5B model on a big GPU is **overhead-bound** (Session 1 §12): differences here are mostly scheduler/Python overhead and won't transfer to a 70B model. Re-run with the model you'll actually ship.

## 11. How to read a vLLM recipe *(reading)*

Since 2026 the recipes live in the main repo as YAML. Each one has the same shape, and knowing it saves you an afternoon per model:

| Key | What's in it |
|---|---|
| `model` | `min_vllm_version`, architecture (dense / MoE), total vs active parameters, context length |
| `features` | Opt-in capabilities as flag lists: `tool_calling`, `reasoning`, `spec_decoding`, `long_context` |
| `variants` | Precision options (BF16, FP8, MXFP4, NVFP4) with the minimum VRAM for each |
| `compatible_strategies` | Which of single-node TP, multi-node TP+PP, TEP (TP+EP) or DEP (DP+EP) are verified |
| `hardware_overrides` | Extra flags per GPU family: Blackwell MoE kernels, ROCm attention backends, Intel XPU |
| `guide` | Prose: install, launch, client usage, accuracy check, troubleshooting |

The flags for the families people host most, copied from those recipes:

| Model | Tool calling | Reasoning | Speculative decoding | Notes from the recipe |
|---|---|---|---|---|
| **gpt-oss-20b / 120b** | `--tool-call-parser openai --enable-auto-tool-choice` | built in (harmony format) | `--speculative-config '{"model":"nvidia/gpt-oss-120b-Eagle3-v3","num_speculative_tokens":7,"method":"eagle3","draft_tensor_parallel_size":1}'` | MXFP4 MoE; 120b fits one 80 GB GPU. H100 TP1 OOM fix: `--gpu-memory-utilization 0.95 --max-num-batched-tokens 1024` |
| **Qwen3 (dense and MoE)** | `--enable-auto-tool-choice --tool-call-parser hermes` | `--reasoning-parser qwen3` | EAGLE-3 speculators from RedHatAI for the large models | 128K context needs YaRN through `--hf-overrides` (§13) |
| **DeepSeek-V3.2** | `--tokenizer-mode deepseek_v32 --tool-call-parser deepseek_v32 --enable-auto-tool-choice` | `--reasoning-parser deepseek_v3` | `--speculative-config '{"method":"mtp","num_speculative_tokens":3}'` (built-in MTP heads) | TP 8, or DP+EP: `-dp 8 --enable-expert-parallel` |
| **Kimi K2 Thinking** | `--enable-auto-tool-choice --tool-call-parser kimi_k2` | `--reasoning-parser kimi_k2` | | TP 8 plus `--trust-remote-code`; `--decode-context-parallel-size 8` for long contexts |
| **Llama 3.3 70B** | (Llama 3 JSON tool format) | | `{"model":"RedHatAI/Llama-3.3-70B-Instruct-speculator.eagle3","num_speculative_tokens":3,"method":"eagle3"}` | One 80 GB GPU in FP8, TP 2–8 in BF16 |

Two patterns to notice. **The parser names are per family, not per model size**, so once Qwen3-8B works, Qwen3-235B needs the same flags plus parallelism. And **speculative decoding is always opt-in**: the recipes treat it as a latency trade you make deliberately (Session 2 §12 shows when it stops paying).

In [ ]:
# Pull any recipe straight from the repo and print the parts you need
import yaml
def recipe(org_model):
    url = f"https://raw.githubusercontent.com/vllm-project/vllm/main/recipes/models/{org_model}.yaml"
    r = yaml.safe_load(requests.get(url, timeout=20).text)
    print(f"{r['meta']['title']}: {r['model']['parameter_count']} total, {r['model'].get('active_parameters', 'dense')} active, "
          f"vLLM >= {r['model']['min_vllm_version']}")
    for name, feat in r.get("features", {}).items():
        print(f"  {name:14} {' '.join(feat['args'])}")
    for name, v in r.get("variants", {}).items():
        print(f"  variant {name:10} {v.get('precision')}, min {v.get('vram_minimum_gb')} GB")
    return r

for m in ["openai/gpt-oss-120b", "Qwen/Qwen3-30B-A3B", "deepseek-ai/DeepSeek-V3.2"]:
    try:
        recipe(m); print()
    except Exception as e:
        print(m, "→", type(e).__name__, "(the recipes path may have moved; browse recipes.vllm.ai)")

## 12. Tool calling end to end *(runnable)*

The server renders your `tools` into the prompt with the chat template, the model emits a call in its own format, and the **tool-call parser** turns that back into OpenAI `tool_calls`. Without `--enable-auto-tool-choice` and the right `--tool-call-parser`, the call arrives as text in `content`.

Qwen2.5 uses the Hermes format, so `hermes` is the parser. The loop below is the whole agent pattern: model asks, you execute, you send the result back, the model answers.

In [ ]:
server = vllm_serve("--enable-auto-tool-choice", "--tool-call-parser", "hermes")
client = OpenAI(base_url=f"{BASE}/v1", api_key="none")

tools = [{"type": "function", "function": {
    "name": "gpu_price", "description": "Hourly on-demand price of a GPU type in USD",
    "parameters": {"type": "object", "properties": {"gpu": {"type": "string", "enum": ["H100", "H200", "B200"]}},
                   "required": ["gpu"]}}}]
PRICES = {"H100": 2.99, "H200": 3.79, "B200": 5.49}                 # illustrative

msgs = [{"role": "user", "content": "What would 10 hours on an H200 cost?"}]
r = client.chat.completions.create(model=MODEL, messages=msgs, tools=tools, tool_choice="auto", max_tokens=200)
call = r.choices[0].message.tool_calls[0]
print("model asked for:", call.function.name, call.function.arguments)

args = json.loads(call.function.arguments)
msgs += [r.choices[0].message.model_dump(exclude_none=True),
         {"role": "tool", "tool_call_id": call.id, "content": json.dumps({"usd_per_hour": PRICES[args["gpu"]]})}]
final = client.chat.completions.create(model=MODEL, messages=msgs, tools=tools, max_tokens=100)
print("answer:", final.choices[0].message.content)

`tool_choice="required"` forces a call, and naming a function (`{"type": "function", "function": {"name": "gpu_price"}}`) forces that one. Both use structured outputs underneath, so the arguments are guaranteed to parse. With `"auto"` the parser is best-effort: small models sometimes emit malformed calls, so validate arguments before executing anything.

In [ ]:
r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Tell me about B200 pricing."}],
                                   tools=tools, tool_choice={"type": "function", "function": {"name": "gpu_price"}}, max_tokens=60)
print(r.choices[0].message.tool_calls[0].function.arguments)
stop(server)

## 13. Reasoning models: separate the thinking from the answer *(runnable)*

Reasoning models write a long chain of thought before answering. The **reasoning parser** splits it out: `message.reasoning` holds the thinking and `message.content` the answer. Your UI shows one, your logs keep both, and your token bill pays for both, which this cell measures.

Qwen3 has a switch in its chat template, `enable_thinking`, passed through `chat_template_kwargs`.

In [ ]:
REASONER = "Qwen/Qwen3-1.7B"
server = vllm_serve("--reasoning-parser", "qwen3", model=REASONER)
client = OpenAI(base_url=f"{BASE}/v1", api_key="none")
q = [{"role": "user", "content": "A 70B model in FP8 on an H100 with 3.35 TB/s: what's the batch-1 decode ceiling in tokens/s?"}]
for thinking in (True, False):
    t0 = time.perf_counter()
    r = client.chat.completions.create(model=REASONER, messages=q, max_tokens=2048,
                                       extra_body={"chat_template_kwargs": {"enable_thinking": thinking}})
    m = r.choices[0].message
    print(f"thinking={thinking}: {r.usage.completion_tokens} tokens, {time.perf_counter() - t0:.1f}s")
    print("  reasoning:", (getattr(m, "reasoning", None) or "")[:160].replace("\n", " "), "...")
    print("  answer   :", (m.content or "")[:200].replace("\n", " "))
stop(server)

Two more controls for reasoning models on the server: `thinking_token_budget` caps the thinking tokens per request, and `reasoning_effort` (`"low"`, `"medium"`, `"high"`) maps to the model's own effort setting where it has one (gpt-oss does). For long-context Qwen3, the recipe enables YaRN like this, and warns it can cost quality below 32K, so only turn it on if you serve long prompts:

```bash
vllm serve Qwen/Qwen3-30B-A3B --max-model-len 131072 \
  --hf-overrides '{"max_position_embeddings":131072,"rope_parameters":{"rope_type":"yarn","rope_theta":1000000,"factor":4.0,"original_max_position_embeddings":32768}}'
```

## 14. Vision-language models *(runnable)*

Images go in as `image_url` content parts (a URL or a base64 data URI). The vision encoder runs during prefill, so an image is effectively a few hundred to a few thousand extra prompt tokens: it shows up in TTFT, not in TPOT. `--limit-mm-per-prompt` caps images per request so one user can't blow up the KV cache.

In [ ]:
VLM = "Qwen/Qwen2.5-VL-3B-Instruct"
server = vllm_serve("--limit-mm-per-prompt", '{"image": 2}', "--max-num-seqs", "16", model=VLM)
client = OpenAI(base_url=f"{BASE}/v1", api_key="none")
img = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/NVIDIA_H100_GPU.jpg/640px-NVIDIA_H100_GPU.jpg"
r = client.chat.completions.create(model=VLM, max_tokens=80, messages=[{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": img}},
    {"type": "text", "text": "What hardware is this? One sentence."}]}])
print(r.choices[0].message.content, f"\nprompt tokens incl. image: {r.usage.prompt_tokens}")
stop(server)

## 15. gpt-oss: the recipe in practice *(runnable on 24 GB+ for 20b, 80 GB for 120b)*

gpt-oss ships its MoE weights in MXFP4, so 120b (117B total, 5.1B active) fits on one 80 GB GPU, and 20b fits on 24 GB cards. It uses OpenAI's *harmony* format; vLLM handles it, and the recipe adds the `openai` tool parser. `reasoning_effort` maps to the model's own low/medium/high setting.

In [ ]:
OSS = "openai/gpt-oss-20b"
server = vllm_serve("--tool-call-parser", "openai", "--enable-auto-tool-choice", model=OSS)
client = OpenAI(base_url=f"{BASE}/v1", api_key="none")
for effort in ("low", "high"):
    t0 = time.perf_counter()
    r = client.chat.completions.create(model=OSS, reasoning_effort=effort, max_tokens=4096,
        messages=[{"role": "user", "content": "How many H100s does a 405B BF16 model need just for weights?"}])
    print(f"effort={effort:5} {r.usage.completion_tokens:5} tokens {time.perf_counter() - t0:5.1f}s  {r.choices[0].message.content[:120]!r}")
stop(server)

For 120b on one GPU the recipe's commands are `vllm serve openai/gpt-oss-120b` (A100/H100) or `--tensor-parallel-size 4` for more throughput. Add the EAGLE-3 draft from §11 for latency-sensitive traffic. gpt-oss also supports the `/v1/responses` API with built-in tools (`--tool-server demo` for the browsing and Python demos).

## 16. Packaging it: Docker, config files, keys and probes *(reading)*

```bash
docker run --gpus all --ipc=host -p 8000:8000 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e HF_TOKEN=$HF_TOKEN \
  vllm/vllm-openai:latest \
  --model Qwen/Qwen3-8B --served-model-name qwen3-8b \
  --api-key $VLLM_API_KEY --config /etc/vllm/serve.yaml
```

- `--ipc=host` (or `--shm-size=16g`): tensor parallelism shares memory between processes; without it you get bus errors.
- Mount the Hugging Face cache, or bake weights into the image, so restarts don't re-download.
- `--served-model-name` decouples what clients call the model from the checkpoint path, so you can swap checkpoints without touching clients.
- `--config serve.yaml` holds the flags as YAML keys (`max-model-len: 32768`), which is easier to review than a 15-flag command line.
- **Probes:** liveness on `GET /health`; readiness after the first successful completion, because `/health` answers before CUDA graphs and compilation have warmed up.
- **Cold start** is weights load + `torch.compile` + CUDA graph capture. vLLM caches compilation in `~/.cache/vllm`; persist that directory between restarts and the second start is much faster.

**On Kubernetes**, use one of the projects built for vLLM rather than a bare Deployment:

| Project | What it adds |
|---|---|
| [vLLM production-stack](https://github.com/vllm-project/production-stack) | Helm chart, prefix-aware router, LMCache integration, Grafana dashboards |
| [llm-d](https://github.com/llm-d/llm-d) | Kubernetes-native scheduler with KV-cache-aware routing and P/D disaggregation (Red Hat, Google, IBM and others) |
| [NVIDIA Dynamo](https://github.com/ai-dynamo/dynamo) | Disaggregation, KV-aware routing and NIXL transfer on top of vLLM, SGLang or TensorRT-LLM |
| [KServe](https://kserve.github.io/website/) | Standard model-serving CRDs with a vLLM runtime |

In [ ]:
# A serve.yaml you can review in a PR; keys are the CLI flags without the leading dashes
cfg = {"model": MODEL, "served-model-name": "assistant", "max-model-len": 8192, "gpu-memory-utilization": 0.9,
       "max-num-seqs": 128, "enable-prefix-caching": True, "kv-cache-dtype": "fp8",
       "enable-auto-tool-choice": True, "tool-call-parser": "hermes", "scheduling-policy": "priority"}
open("serve.yaml", "w").write(yaml.safe_dump(cfg, sort_keys=False))
print(open("serve.yaml").read())
print("vllm serve --config serve.yaml")

## 17. Operating it: priorities, metrics and a launch checklist

**Priority scheduling.** With `--scheduling-policy priority`, each request can carry a `priority` (lower runs first). Interactive traffic and batch jobs then share one deployment: batch soaks up idle capacity and gets pre-empted when users arrive.

In [ ]:
server = vllm_serve("--scheduling-policy", "priority", "--max-num-seqs", "8")
from openai import AsyncOpenAI
ac = AsyncOpenAI(base_url=f"{BASE}/v1", api_key="none")
async def one(prio, i):
    t0 = time.perf_counter()
    await ac.chat.completions.create(model=MODEL, max_tokens=128, extra_body={"priority": prio, "ignore_eos": True},
                                     messages=[{"role": "user", "content": f"Write about GPUs #{i}"}])
    return prio, time.perf_counter() - t0
res = await asyncio.gather(*[one(10, i) for i in range(48)], *[one(0, 100 + i) for i in range(8)])
df_ = pd.DataFrame(res, columns=["priority", "latency_s"])
display(df_.groupby("priority").latency_s.describe()[["count", "50%", "max"]].round(2))
stop(server)

**What to alert on** (all on `/metrics`):

| Metric | Alert when | Usually means |
|---|---|---|
| `vllm:num_requests_waiting` | Above 0 for minutes | Out of capacity; scale out (this is the autoscaling signal, not GPU utilization) |
| `vllm:kv_cache_usage_perc` | Near 1.0 | Pre-emptions coming; lower `--max-num-seqs` or add memory (FP8 KV, more GPUs) |
| `vllm:num_preemptions` | Rising | KV cache too small for the concurrency you admit |
| TTFT p99 | Above SLO | Long prompts or queueing; check waiting count first |
| Prefix cache hits ÷ queries | Drops after a deploy | Someone put a timestamp or request ID at the top of the system prompt |

**Launch checklist for a new open-weight model**

1. Read its recipe (§11); note the parsers, the variant for your GPU and `min_vllm_version`.
2. Size it (§4) for your real context length and concurrency, not the model's maximum context.
3. Start with the recipe command, then set `--max-model-len` to what you need.
4. Check the start-up log: KV cache size in tokens and "maximum concurrency" for your max-model-len.
5. Test tool calls and reasoning output through the OpenAI client (§12–13) before any load test.
6. Load-test with your prompt and output lengths: `vllm bench serve` (Session 2 §22) or the harness in Session 2 §27.
7. Evaluate quality at the precision you'll serve (Session 2 §17, plus a task eval).
8. Then tune: prefix-cache-friendly prompts, FP8 KV, spec decode for latency, DP replicas for throughput.

### Where practitioners share working set-ups

- **vLLM recipes** (`recipes/` in the vLLM repo, [recipes.vllm.ai](https://recipes.vllm.ai)): maintainer-verified commands per model and GPU, with troubleshooting sections. The best first stop.
- **vLLM blog** ([blog.vllm.ai](https://blog.vllm.ai)): launch-day posts for major models (gpt-oss, DeepSeek, Qwen3, Kimi) with the reasoning behind the flags.
- **Aleksa Gordić, [Inside vLLM](https://www.aleksagordic.com/blog/vllm)**: how the engine works internally, from the scheduler to the KV cache manager.
- **[Modal examples](https://github.com/modal-labs/modal-examples)** and **[SkyPilot `llm/` examples](https://github.com/skypilot-org/skypilot/tree/master/llm)**: complete, runnable deployments of open-weight models on cloud GPUs.
- **[r/LocalLLaMA](https://www.reddit.com/r/LocalLLaMA/)** and X: fast, anecdotal, often the first place a working flag combination for a new model appears. Treat the numbers there as hypotheses and re-measure (§27 collects some).

In [ ]:
print(gpu_mem())

## 18. Sleep mode: give the GPU back without killing the engine

`LLM(..., enable_sleep_mode=True)` lets you free GPU memory between jobs and get it back in seconds instead of a full restart.

| Call | Weights | KV cache | Use it when |
|---|---|---|---|
| `llm.sleep(level=1)` | Moved to CPU RAM | Discarded | Same model wakes up later (alternating jobs on one GPU) |
| `llm.sleep(level=2)` | Discarded | Discarded | New weights are coming (RLHF: the trainer updates them) |
| `llm.wake_up(tags=["weights"])`, then `tags=["kv_cache"]` | | | Wake in two steps to keep peak memory low during a weight update |

RL frameworks (verl, OpenRLHF, TRL) use this to share GPUs between the trainer and vLLM rollouts. On the server it's `POST /sleep?level=1` and `POST /wake_up`, available when `VLLM_SERVER_DEV_MODE=1`.

In [ ]:
llm = LLM(model=MODEL, enable_sleep_mode=True, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)
print("awake  :", gpu_mem())
t0 = time.perf_counter(); llm.sleep(level=1); print(f"asleep : {gpu_mem()}   (sleep took {time.perf_counter() - t0:.1f}s)")
t0 = time.perf_counter(); llm.wake_up(); print(f"awake  : {gpu_mem()}   (wake took {time.perf_counter() - t0:.1f}s)")
print(llm.generate(["Still works:"], SamplingParams(max_tokens=8), use_tqdm=False)[0].outputs[0].text)
shutdown(llm); del llm

## 19. `AsyncLLM`: the engine inside the server, in your own process

`vllm serve` is a thin FastAPI layer over `AsyncLLM`. Use it directly when you're building something that isn't an HTTP server: an agent loop, a streaming evaluation harness, a Ray actor. `RequestOutputKind.DELTA` gives only the new tokens on each iteration.

In [ ]:
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.sampling_params import RequestOutputKind
from vllm.v1.engine.async_llm import AsyncLLM

engine = AsyncLLM.from_engine_args(AsyncEngineArgs(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6))

async def stream(prompt, rid):
    sp = SamplingParams(max_tokens=60, temperature=0.7, output_kind=RequestOutputKind.DELTA)
    first, n, t0 = None, 0, time.perf_counter()
    async for out in engine.generate(prompt, sp, request_id=rid):
        if out.outputs[0].text:
            first = first or time.perf_counter(); n += len(out.outputs[0].token_ids)
    return rid, (first - t0) * 1e3, n

res = await asyncio.gather(*[stream(f"Fact {i} about GPUs:", f"req-{i}") for i in range(16)])
display(pd.DataFrame(res, columns=["request", "ttft_ms", "tokens"]).round(0).head())
engine.shutdown()

## 20. Reaching into the workers: `apply_model` and `collective_rpc`

Two escape hatches for debugging and research. `llm.apply_model(fn)` runs `fn(model)` on the `nn.Module` in every worker; `llm.collective_rpc(method)` calls a worker method on every rank. Passing Python functions to other processes needs `VLLM_ALLOW_INSECURE_SERIALIZATION=1` (it uses pickle), so keep it to notebooks and trusted code.

In [ ]:
os.environ["VLLM_ALLOW_INSECURE_SERIALIZATION"] = "1"
llm = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)

def describe(model):
    n = sum(p.numel() for p in model.parameters())
    kinds = sorted({type(m).__name__ for m in model.modules() if "Linear" in type(m).__name__})
    return f"{n / 1e9:.2f}B parameters on this rank; linear layer types: {kinds}"
print(llm.apply_model(describe))
shutdown(llm); del llm

The layer names (`QKVParallelLinear`, `MergedColumnParallelLinear`, `RowParallelLinear`) show how vLLM fuses Q, K and V into one matmul and shards layers for tensor parallelism, the same picture as the deck's parallelism slide.

## 21. Batch jobs: `vllm run-batch` and data parallelism

For offline jobs over files you don't need a server. `vllm run-batch` reads the **OpenAI Batch API** JSONL format, so the same file works against OpenAI and against your own GPU:

In [ ]:
rows = [{"custom_id": f"row-{i}", "method": "POST", "url": "/v1/chat/completions",
         "body": {"model": MODEL, "max_tokens": 32,
                  "messages": [{"role": "user", "content": f"One sentence on GPU fact #{i}."}]}} for i in range(200)]
with open("batch_in.jsonl", "w") as f:
    f.write("\n".join(json.dumps(r) for r in rows))
t0 = time.time()
subprocess.run(["vllm", "run-batch", "-i", "batch_in.jsonl", "-o", "batch_out.jsonl", "--model", MODEL,
                "--max-model-len", str(MAX_LEN)], check=True, capture_output=True)
outs = [json.loads(l) for l in open("batch_out.jsonl")]
print(f"{len(outs)} rows in {time.time() - t0:.0f}s (includes engine start-up)")
print(outs[0]["response"]["body"]["choices"][0]["message"]["content"])

**Several GPUs, one model that fits on each:** run data parallel. The server does it with one flag, `vllm serve MODEL --data-parallel-size 4`, and load-balances internally. Offline, the pattern in vLLM's `examples/features/data_parallel/data_parallel_offline.py` starts one process per rank and sets `VLLM_DP_RANK`, `VLLM_DP_SIZE`, `VLLM_DP_MASTER_IP` and `VLLM_DP_MASTER_PORT` before creating `LLM`. For many nodes, Ray Data's LLM integration (`ray.data.llm`) shards a dataset across vLLM replicas for you.

Rule of thumb: **data parallel when the model fits, tensor parallel when it doesn't.** DP gives near-linear throughput with no inter-GPU traffic; TP buys memory and latency at the cost of an all-reduce every layer.

## 22. Profiling: where does a step actually go? *(reading)*

When a number surprises you, look at the kernel timeline. vLLM wraps the PyTorch profiler:

```python
# vLLM >= 0.13: pass a profiler config (CLI: --profiler-config '{"profiler": "torch", "torch_profiler_dir": "./vllm_profile"}')
llm = LLM(model=MODEL, profiler_config={"profiler": "torch", "torch_profiler_dir": "./vllm_profile"})
# older releases used the env var instead: os.environ["VLLM_TORCH_PROFILER_DIR"] = "./vllm_profile" before LLM(...)
llm.start_profile()
llm.generate(["Hello"] * 32, SamplingParams(max_tokens=64))
llm.stop_profile()          # open the trace in https://ui.perfetto.dev
```

What to look for: long gaps between kernels (overhead regime, so check CUDA graphs are on), attention vs GEMM share (long context → attention-dominated), and NCCL all-reduce time with tensor parallelism. For whole-system views use Nsight Systems: `nsys profile -o trace vllm serve …`.

"Can I run model X on machine Y?" has a two-line answer: **does it fit**, and **how fast does it stream**. Both come from Session 1 §4's arithmetic, with three twists for local hardware: MoE (memory ∝ total params, speed ∝ active params), offload (part of the model lives on a slower tier), and quantization formats (GGUF, AWQ, FP8, MXFP4).

## 23. A planner that reads any Hugging Face config *(no GPU needed)*

In [ ]:
from huggingface_hub import hf_hub_download, HfApi

def model_facts(repo):
    # Pull config.json + safetensors header; works for dense, GQA, MLA and MoE models.
    cfg = json.load(open(hf_hub_download(repo, "config.json")))
    cfg = cfg.get("text_config", cfg)                                       # multimodal wrappers
    try:
        counts = HfApi().get_safetensors_metadata(repo).parameter_count     # {dtype: count}
        # 4-bit formats (MXFP4) are stored packed as U8: two weights per byte (block scales add ~5%)
        params = sum(n * (2 if dt == "U8" else 1) for dt, n in counts.items())
    except Exception:
        params = None
    L = cfg["num_hidden_layers"]
    if "kv_lora_rank" in cfg:                                               # MLA (DeepSeek-V2/V3, Kimi K2)
        kv_elems = L * (cfg["kv_lora_rank"] + cfg.get("qk_rope_head_dim", 64))
    else:
        hd = cfg.get("head_dim") or cfg["hidden_size"] // cfg["num_attention_heads"]
        kv_elems = 2 * L * cfg.get("num_key_value_heads", cfg["num_attention_heads"]) * hd
    experts = cfg.get("num_local_experts") or cfg.get("n_routed_experts") or cfg.get("num_experts")
    active = cfg.get("num_experts_per_tok") or cfg.get("num_experts_per_token")
    return {"repo": repo, "params_B": params / 1e9 if params else None, "layers": L,
            "kv_bytes_per_token_bf16": kv_elems * 2, "experts": experts, "experts_per_token": active,
            "sliding_window": cfg.get("sliding_window"), "max_pos": cfg.get("max_position_embeddings")}

for repo in ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen3-30B-A3B", "openai/gpt-oss-120b"]:
    try:
        f = model_facts(repo); print({k: (round(v, 1) if isinstance(v, float) else v) for k, v in f.items()})
    except Exception as e:
        print(repo, "→", type(e).__name__, "(gated? run `huggingface-cli login`)")

In [ ]:
HW = {  # name: (fast memory GB, fast BW GB/s, slow memory GB, slow BW GB/s)  slow = system RAM for offload
    "RTX 4090 + 64GB DDR5":   (24, 1000, 64, 60),
    "RTX 5090 + 128GB DDR5":  (32, 1790, 128, 70),
    "2x RTX 4090 (PCIe)":     (48, 2000, 64, 60),
    "Mac M3 Ultra 512GB":     (512 * 0.75, 819, 0, 1),   # macOS lets the GPU wire ~75% by default
    "DGX Spark 128GB":        (128 * 0.9, 273, 0, 1),
    "1x H100 80GB":           (80, 3350, 0, 1),
    "8x H100 (TP=8)":         (640, 26800, 0, 1),
}
BYTES = {"BF16": 2.0, "FP8/Q8": 1.06, "Q4 (GGUF Q4_K_M / AWQ)": 0.56, "MXFP4": 0.53}

def plan(total_B, active_B, kv_kib_per_tok, ctx=32_768, users=1, fmt="Q4 (GGUF Q4_K_M / AWQ)", kv_bytes=1):
    w = total_B * BYTES[fmt]; kv = kv_kib_per_tok * 1024 * ctx * users * kv_bytes / 2 / 1e9
    rows = []
    for name, (fast, fbw, slow, sbw) in HW.items():
        need = w + kv + 1.5                                                 # +1.5 GB activations/runtime
        on_gpu = min(need, fast); spill = need - on_gpu
        fits = spill <= 0 or spill <= slow
        # bytes streamed per token: active weights (+ KV); spilled share comes from the slow tier
        act = active_B * BYTES[fmt]; frac_spill = max(0, spill - kv) / max(w, 1e-9)
        t = act * (1 - frac_spill) / fbw + act * frac_spill / sbw          # seconds per token (ceiling)
        rows.append({"hardware": name, "needs_GB": round(need), "offloaded_GB": round(max(spill, 0)),
                     "fits": "yes" if spill <= 0 else ("with offload" if fits else "NO"),
                     "decode_ceiling_tok_s": round(1 / t) if fits else None})
    return pd.DataFrame(rows)

print("Qwen3-235B-A22B, Q4, 32K context, FP8 KV:")
display(plan(235, 22, kv_kib_per_tok=188, fmt="Q4 (GGUF Q4_K_M / AWQ)"))
print("gpt-oss-120b, native MXFP4:")
display(plan(117, 5.1, kv_kib_per_tok=72, fmt="MXFP4"))
print("Llama-3.3-70B dense, Q4:")
display(plan(70, 70, kv_kib_per_tok=320))

> Two patterns jump out. **MoE + offload is surprisingly usable** because each token only touches the active experts. **Dense 70B on a single consumer GPU with offload is painful**: every token drags tens of GB across a ~60 GB/s RAM bus. These are *ceilings*; real numbers are typically 50–80% of them, and **prefill suffers far more from offload than decode** because the CPU is a poor matmul engine.

## 24. The local toolbox *(reading)*

| Tool | Best at | Key flags / commands |
|---|---|---|
| **llama.cpp** (`llama-server`, `llama-bench`) | Anything that doesn't fit in VRAM; GGUF quants; CPU + GPU split; Mac/AMD/Intel/Vulkan | `-ngl 99` (layers on GPU), `--n-cpu-moe N` (keep N layers' experts on CPU), `-ot "exps=CPU"`, `-c` ctx, `-fa on`, `--cache-type-k/v q8_0` |
| **Ollama / LM Studio** | Friendly wrappers around llama.cpp (and MLX in LM Studio) | `OLLAMA_FLASH_ATTENTION=1`, `OLLAMA_KV_CACHE_TYPE=q8_0`, `num_ctx` |
| **MLX / mlx-lm** | Apple Silicon, often faster than llama.cpp on Macs | `mlx_lm.generate`, `mlx_lm.server`, 4-bit `mlx-community/*` repos |
| **exo** | Pooling several Macs (Thunderbolt 5) for 1T-parameter MoEs | — |
| **KTransformers** | Big MoE with experts on CPU (AMX) + attention on GPU; strong prefill in hybrid mode | — |
| **vLLM / SGLang** | Multi-user serving on NVIDIA/AMD, even on 1–4 consumer GPUs | `--cpu-offload-gb`, `--enforce-eager`, `--max-model-len`, `--kv-cache-dtype fp8`, `--pipeline-parallel-size` |

In [ ]:
# llama.cpp reports prefill and decode separately: pp = prompt processing (prefill), tg = token generation (decode)
#   brew install llama.cpp        (macOS)   or download a release build from github.com/ggml-org/llama.cpp
LLAMA_CMDS = r'''
# 1) Measure prefill vs decode for a GGUF model, all layers on GPU
llama-bench -hf Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M -ngl 99 -p 512,4096 -n 128 -fa 1

# 2) Serve a big MoE that does NOT fit in VRAM: attention + shared layers on GPU, experts on CPU
llama-server -hf unsloth/Qwen3-235B-A22B-Instruct-2507-GGUF:Q4_K_M \
    -ngl 99 --n-cpu-moe 70 -c 32768 -fa on \
    --cache-type-k q8_0 --cache-type-v q8_0 --port 8080
#    lower --n-cpu-moe until you OOM, then step back up: every expert layer moved to GPU speeds up both phases

# 3) Same idea with an explicit tensor override (older builds)
llama-server -m model.gguf -ngl 99 -ot ".ffn_.*_exps.=CPU" -c 16384

# 4) Apple Silicon with MLX
pip install mlx-lm
mlx_lm.generate --model mlx-community/Qwen3-30B-A3B-4bit --prompt "Hello" --max-tokens 200
mlx_lm.server  --model mlx-community/Qwen3-30B-A3B-4bit --port 8080
'''
print(LLAMA_CMDS)
import shutil
if shutil.which("llama-bench"):
    !llama-bench -hf Qwen/Qwen2.5-1.5B-Instruct-GGUF:Q4_K_M -ngl 99 -p 512 -n 128

> Every `llama-server` endpoint is OpenAI-compatible too, so Session 2 §27's `bench_openai("http://localhost:8080", "any")` benchmarks it with the same harness.

## 25. vLLM when the model *almost* fits

Three levers, cheapest first: shrink the KV cache (`--max-model-len`, FP8 KV), skip CUDA-graph memory (`--enforce-eager`), then spill weights to CPU (`--cpu-offload-gb`, slow: every step crosses PCIe).

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL, max_model_len=2048, gpu_memory_utilization=0.5,
          kv_cache_dtype="fp8", enforce_eager=True,
          cpu_offload_gb=1)          # pretend the GPU is tiny: 1 GB of weights lives in CPU RAM
t0 = time.perf_counter()
o = llm.generate(["Explain CPU offload in one paragraph."], SamplingParams(max_tokens=128, temperature=0))
n = len(o[0].outputs[0].token_ids); print(o[0].outputs[0].text[:300])
print(f"\n{n / (time.perf_counter() - t0):.0f} tok/s with offload. Compare with Session 1 §12's number: that gap is PCIe.")
shutdown(llm)

## 26. Edge-case checklist: errors you will hit *(reading)*

| Symptom / error | Cause | Fix |
|---|---|---|
| `CUDA out of memory` at startup | Memory fraction too high, or another process on the GPU | `nvidia-smi` to find it; lower `--gpu-memory-utilization` / `--mem-fraction-static` |
| "max_model_len … larger than the maximum number of tokens that can be stored in KV cache" | Model's default context (often 128K) doesn't fit | `--max-model-len 32768`, FP8 KV cache, or more GPUs |
| `Bfloat16 is only supported on GPUs with compute capability ≥ 8.0` | T4 / V100 / older | `--dtype half` |
| FP8 / NVFP4 checkpoint won't load | FP8 needs sm_89+ (Ada/Hopper); FP4 needs Blackwell | Use AWQ/GPTQ INT4 or the BF16 checkpoint |
| `trust_remote_code` error | New architecture with custom modelling code | Add `--trust-remote-code` *after reading the code*, or upgrade the engine |
| Garbage or repeated output | Wrong chat template / tokenizer, or base model instead of instruct | Use `llm.chat` / chat endpoint; pass `--chat-template`; check the repo is `-Instruct` |
| Multi-GPU hangs on consumer cards | NCCL P2P over PCIe is broken on some boards | `NCCL_P2P_DISABLE=1`; prefer `--pipeline-parallel-size` over TP without NVLink |
| Docker multi-GPU crash (`bus error`, shm) | Tiny `/dev/shm` | `--ipc=host` or `--shm-size=16g` |
| 401 / 403 downloading weights | Gated model (Llama, Gemma) | Accept licence on HF; `huggingface-cli login` or `HF_TOKEN` |
| Download takes forever / fills disk | Pulling every quant in a GGUF repo | `huggingface-cli download repo --include "*Q4_K_M*"`; `HF_HUB_ENABLE_HF_TRANSFER=1` |
| First request very slow | Compile + CUDA graph capture + cold page cache | Warm-up requests; persist compile cache; keep weights on local NVMe |
| Mac: model "fits" but crashes | GPU can only wire ~75% of unified memory by default | `sudo sysctl iogpu.wired_limit_mb=<MB>` (at your own risk) |
| Long prompts slow with MoE offload | Prefill runs on CPU for offloaded experts | More expert layers on GPU, bigger `-ub`/`-b` batch, or KTransformers |
| Vision model OOMs on images | Encoder activations + many image tokens | Limit `--limit-mm-per-prompt`, lower image resolution, smaller `max_num_seqs` |
| Throughput collapses at long context | KV cache full → preemption / recompute | Watch `num_preemptions`; FP8 KV; fewer concurrent seqs |

## 27. What practitioners report (treat as anecdotes, not benchmarks) *(reading)*

| Setup | Reported result | Source |
|---|---|---|
| 4× Mac Studio M3 Ultra (exo, Thunderbolt 5), Kimi K2 Thinking (~1T MoE) | ~25–28 tok/s decode, <500 W | Jeff Geerling via [heise](https://heise.de/-11085716) |
| 12 GB GPU, Qwen3 MoE, raising `--n-cpu-moe` tuning | Decode 17 → 34 tok/s after re-tuning offload | [r/LocalLLaMA summary](https://aiweekly.co/alerts/qwen3-moe-cpu-expert-offload-doubles-decode-speed-on-12gb-vram) |
| Experts on CPU vs GPU, 26B MoE | Prefill ~5.3× slower, decode ~2.4× slower | [ai-muninn](https://ai-muninn.com/en/blog/run-119b-moe-on-3x-2080ti) |
| M5 MacBook Pro, Qwen3.5-9B | llama.cpp 22, MLX 25, uzu 92 tok/s | [Avi Chawla on X](https://x.com/_avichawla/status/2115010493899502058) |
| Llama 4 Maverick, hybrid CPU/GPU | Decode 25 tok/s but prefill only 18 tok/s | r/LocalLLaMA thread |

The pattern across all of them: **local decode is a memory-bandwidth story, local prefill is a compute story**, and offload hurts prefill most.

## Exercises

1. **Check the arithmetic.** In §1, make the tensor-parallel split uneven (3 GPUs for a 4096-wide layer). What has to change?
2. **Measure your links.** Run the all-reduce benchmark in §2 with and without `NCCL_P2P_DISABLE=1`. What does that tell you about TP on this machine?
3. **TP or DP?** With two GPUs, compare TP=2 and DP=2 at concurrency 1 and 64. Which would you deploy for a chatbot, and which for a batch job?
4. **Plan a model you want.** Pick a model from your backlog, read its recipe, size it, and write the launch command for your hardware.
5. **Write the runbook.** From the production section, list the three alerts you'd page on and what each one means.

In [ ]:
# Stop anything this notebook started
for _name in ("server", "sg", "trt"):
    if isinstance(globals().get(_name), subprocess.Popen):
        stop(globals()[_name])
print(gpu_mem())

End of the series. The README in the project root maps every topic to its slides and notebook sections, with the reading list.